In [ ]:
import numpy
import pandas
from torch.utils.tensorboard import SummaryWriter
import wandb
import glob
from PIL import Image
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset, WeightedRandomSampler
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.nn as nn
from torchvision import models
from tqdm.notebook import tqdm
from transformers import ViTForImageClassification, AutoImageProcessor
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import seaborn as sns

In [ ]:
train_df = pd.read_csv('../input/datasets/alessandrasala79/ai-vs-human-generated-dataset/train.csv')
test_df = pd.read_csv('../input/datasets/alessandrasala79/ai-vs-human-generated-dataset/test.csv')

### **Load Back Model VIA Huggingface**

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Login
secrets = UserSecretsClient()
hf_token = secrets.get_secret("Model_Writer")
login(token=hf_token)

# Push everything to HF
model.push_to_hub("gazeng/vit-ai-detector_3")
feature_extractor.push_to_hub("gazeng/vit-ai-detector_3")

print("Done! Check https://huggingface.co/your-username/vit-ai-detector")

### **Transformation**

In [ ]:
# Add the full address to file name, so it becomes something like /content/drive/MyDrive/172b_data/train_data/abacjbcahaksncaj.jpg
dir = "../input/datasets/alessandrasala79/ai-vs-human-generated-dataset/"
train_df['file_name'] = train_df['file_name'].apply(lambda x: os.path.join(dir, x))
# Make sure the label is a string, don't want it to be treated as integer
train_df['label'] = train_df['label'].astype(str)
train_df = train_df.drop(columns='Unnamed: 0')

In [ ]:
print(f"Train set shape: {train_df.shape}")
print(f"Test set shape: {test_df.shape}")

print("\n--- Train Data Columns ---")
print(train_df.columns.tolist())
print("\n--- Test Data Columns ---")
print(test_df.columns.tolist())

print("\n--- Train Data Preview ---")
display(train_df.head())
# check label count
print("\n--- Train Data Label Count ---")
print(train_df['label'].value_counts())

### **Training Split**

In [ ]:
from sklearn.model_selection import StratifiedKFold

In [ ]:
def folds(dataset):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=100)
    dataset["fold"] = -1
    
    X = dataset["file_name"].values
    y = dataset["label"].values
    
    for fold, (_, val_idx) in enumerate(skf.split(X, y)):
        dataset.loc[val_idx, "fold"] = fold
    print(dataset["fold"].value_counts().sort_index())
    print("\nLabel distribution per fold (counts):")
    print(pd.crosstab(dataset["fold"], dataset["label"]))
    
    print("\nLabel distribution per fold (row proportions):")
    print(pd.crosstab(dataset["fold"], dataset["label"], normalize="index"))
    return dataset

train_df = folds(train_df)

In [ ]:
def k(dataset, k=0):
    train_split = dataset[dataset["fold"] != k].reset_index(drop=True)
    val_split   = dataset[dataset["fold"] == k].reset_index(drop=True)
    
    print(train_split.shape, val_split.shape)
    return train_split, val_split

train_split, val_split = k(train_df)

In [ ]:
folder = [
    "/kaggle/input/datasets/renhuang8/genimage-subset-detection/gan_pool/*.png",
    "/kaggle/input/datasets/renhuang8/genimage-subset-detection/mj_pool/*.png",
    "/kaggle/input/datasets/renhuang8/genimage-subset-detection/real_pool/*.JPEG",
    "/kaggle/input/datasets/renhuang8/genimage-subset-detection/sd_pool/*.png"
]

def images(data, label, title, boo = True):
    image = glob.glob(data, recursive=True)
    image_df = pd.DataFrame(image, columns=["file_name"])
    image_df["label"] = label
    image_df["source_pool"] = title
    if(boo):
        val_ds   = BinaryImageDataset(image_df,   transform=val_tfms,   is_test=False)
        loader = DataLoader(
            val_ds,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=torch.cuda.is_available(),
        )
        x, y = next(iter(loader))
        return image_df, loader
    return image_df

In [ ]:
def split_and_get_sampler(df, label_col='label', source_col='source_pool',
                           train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42):
    """
    Splits a DataFrame into train/val/test (stratified) and returns a weighted sampler for training.

    Returns:
        train_df, val_df, test_df, train_sampler
    """
    # --- Stratified split ---
    train_df, temp_df = train_test_split(
        df,
        stratify=df[label_col],
        test_size=(1 - train_ratio),
        random_state=seed
    )
    
    val_size_adjusted = val_ratio / (val_ratio + test_ratio)
    val_df, test_df = train_test_split(
        temp_df,
        stratify=temp_df[label_col],
        test_size=(1 - val_size_adjusted),
        random_state=seed
    )
    
    # Reset indices
    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    
    # --- Weighted sampler for training ---
    # Weight by source pool so each generator/source is equally represented
    source_counts = train_df[source_col].value_counts()
    weights = train_df[source_col].map(lambda x: 1.0 / source_counts[x])
    
    train_sampler = WeightedRandomSampler(
        weights=weights.values,
        num_samples=len(weights),
        replacement=True
    )
    
    return train_df, val_df, test_df, train_sampler

In [ ]:
gan_df = images(folder[0], "1", "gan", False)
mj_df = images(folder[1], "1", "mj", False)
real_df = images(folder[2], "0", "real", False)
sd_df = images(folder[3], "1", "sd", False)
pool_df = pd.concat([gan_df, mj_df, real_df, sd_df,  train_df])

train_split, val_split, test_split, sampler = split_and_get_sampler(
    pool_df,
    label_col='label',
    source_col='source_pool'
)

### **Loader**

In [ ]:
class BinaryImageDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None, is_test: bool = False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test
        if not self.is_test:
            if "label" not in self.df.columns:
                raise ValueError("Training/validation df must include a 'label' column.")
        if "file_name" not in self.df.columns:
            raise ValueError("df must include a 'file_name' column with image paths.")
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx: int):
        path = self.df.iloc[idx]["file_name"]
        # Load image
        img = Image.open(path).convert("RGB")
        # Apply transforms
        if self.transform is not None:
            img = self.transform(img)
        if self.is_test:
            if "label" in self.df.columns:
                y_str = self.df.loc[idx, "label"]
                y = torch.tensor([float(y_str)], dtype=torch.float32)
            else:
                y = torch.tensor([-1.0], dtype=torch.float32)
            return img, y

        # label in CSV is "0"/"1" (string) -> float target for BCE
        y_str = self.df.loc[idx, "label"]
        y = torch.tensor([float(y_str)], dtype=torch.float32)  # shape [1]
        return img, y

In [ ]:
def get_weighted_sampler(df: pd.DataFrame, source_col: str = 'source_pool') -> WeightedRandomSampler:
    """
    Creates a WeightedRandomSampler that balances sampling across source pools.
    Each source/generator contributes equally regardless of its size in the dataset.

    Args:
        df:         DataFrame containing a source pool column.
        source_col: Column name identifying the data source/generator.

    Returns:
        WeightedRandomSampler with per-sample weights based on inverse source frequency.
    """
    source_counts = df[source_col].value_counts()
    weights = df[source_col].map(lambda x: 1.0 / source_counts[x])

    return WeightedRandomSampler(
        weights=weights.to_numpy(dtype=float),
        num_samples=len(weights),
        replacement=True
    )

In [ ]:
def create_dataloaders(train_df, val_df, test_df=None, batch_size=32, num_workers=0):
    """
    Creates DataLoaders from provided DataFrames.
    
    Args:
        train_df: DataFrame with 'file_name' and 'label' columns
        val_df:   DataFrame with 'file_name' and 'label' columns
        test_df:  Optional DataFrame (no label required)
        batch_size: Batch size for all loaders
        num_workers: Number of workers for DataLoader
    
    Returns:
        train_loader, val_loader, (test_loader or None)
    """
    IMG_SIZE = 224
    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD  = [0.229, 0.224, 0.225]

    train_tfms = transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02),
        transforms.RandomGrayscale(p=0.05),
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
        transforms.RandomAdjustSharpness(sharpness_factor=2, p=0.3),
        transforms.ToTensor(),
        transforms.RandomApply([transforms.Lambda(lambda x: x + torch.randn_like(x) * 0.02)], p=0.5),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

    val_tfms = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

    pin_memory = torch.cuda.is_available()

    # Train loader
    train_df = train_df.reset_index(drop=True)
    train_ds = BinaryImageDataset(train_df, transform=train_tfms, is_test=False)
    sampler  = get_weighted_sampler(train_df)
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                              num_workers=num_workers, pin_memory=pin_memory)

    # Val loader
    val_ds = BinaryImageDataset(val_df, transform=val_tfms, is_test=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            num_workers=num_workers, pin_memory=pin_memory)

    # Optional test loader
    test_loader = None
    if test_df is not None:
        test_ds = BinaryImageDataset(test_df, transform=val_tfms, is_test=True)
        test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                                 num_workers=num_workers, pin_memory=pin_memory)

    # Sanity check
    x, y = next(iter(test_loader))
    print(f"x: {x.shape} {x.dtype}")
    print(f"y: {y.shape} {y.dtype}")
    print(f"y unique (sample): {torch.unique(y)[:10]}")
    print(f"\nSplit sizes → Train: {len(train_df)} | Val: {len(val_df)}" + 
          (f" | Test: {len(test_df)}" if test_df is not None else ""))

    return train_loader, val_loader, test_loader

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(
    train_split, val_split, test_df=test_split
)

### **Model 1**

In [ ]:
# ----------------------------
# Model
# ----------------------------
MODEL_NAME = 'google/vit-base-patch16-224'
feature_extractor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = ViTForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    ignore_mismatched_sizes=True
)

# Freeze all backbone layers except last 4 transformer blocks
for name, param in model.named_parameters():
    param.requires_grad = False  # freeze everything first

for name, param in model.named_parameters():
    if any(f"encoder.layer.{i}" in name for i in range(8, 12)):  # unfreeze last 4
        param.requires_grad = True

# Unfreeze classifier head always
for param in model.classifier.parameters():
    param.requires_grad = True

# Replace classifier head with stronger dropout
model.classifier = nn.Sequential(
    nn.LayerNorm(model.config.hidden_size),
    nn.Dropout(p=0.5),
    nn.Linear(model.config.hidden_size, 256),
    nn.GELU(),
    nn.Dropout(p=0.3),
    nn.Linear(256, 1)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# ----------------------------
# Optimizer — separate LR for backbone vs head
# ----------------------------
backbone_params = [p for n, p in model.named_parameters() 
                   if p.requires_grad and "classifier" not in n]
head_params     = [p for n, p in model.named_parameters() 
                   if p.requires_grad and "classifier" in n]

optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": 1e-5},   # lower LR for backbone
    {"params": head_params,     "lr": 1e-4},   # higher LR for head
], weight_decay=0.05)

# ----------------------------
# Scheduler + criterion
# ----------------------------
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)
criterion = nn.BCEWithLogitsLoss()

# ----------------------------
# run_epoch with label smoothing
# ----------------------------
def smooth_labels(labels, smoothing=0.1):
    return labels * (1 - smoothing) + 0.5 * smoothing

def run_epoch(model, loader, optimizer, criterion, training, threshold=0.5, grad_clip=2.0):
    model.train() if training else model.eval()
    total_loss, total_acc = 0.0, 0.0
    with torch.set_grad_enabled(training):
        loop = tqdm(loader, desc="Train" if training else "Val", leave=False)
        for pixel_values, labels in loop:
            pixel_values = pixel_values.to(device)
            labels = labels.float().to(device).squeeze(1)

            # Forward pass first
            outputs = model(pixel_values=pixel_values)
            logits  = outputs.logits.squeeze(1)

            # Loss: smoothed for training, original for val
            if training:
                smoothed = smooth_labels(labels, smoothing=0.1)
                loss = criterion(logits, smoothed)
            else:
                loss = criterion(logits, labels)

            # Accuracy always on original labels
            acc = binary_accuracy(logits, labels, threshold)

            if training:
                optimizer.zero_grad()
                loss.backward()
                if grad_clip:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()

            total_loss += loss.item()
            total_acc  += acc
            loop.set_postfix(loss=loss.item(), acc=acc)

    n = len(loader)
    return total_loss / n, total_acc / n

def binary_accuracy(logits: torch.Tensor, labels: torch.Tensor, threshold: float = 0.5) -> float:
    preds = (torch.sigmoid(logits) >= threshold).float()
    return (preds == labels).float().mean().item()

In [ ]:
NUM_EPOCHS   = 15
best_val_acc = 0.0
SAVE_PATH    = "/kaggle/working/vit-ai-detector-best"

for epoch in range(NUM_EPOCHS):
    THRESHOLD = 0.65 
    train_loss, train_acc = run_epoch(model, train_loader, optimizer, criterion, training=True, threshold=THRESHOLD)
    val_loss,   val_acc   = run_epoch(model, val_loader,  optimizer, criterion, training=False, threshold=THRESHOLD)
    scheduler.step()
    writer.add_scalar("Loss/train", train_loss, epoch)
    writer.add_scalar("Loss/val",   val_loss,   epoch)
    writer.add_scalar("Accuracy/train", train_acc, epoch)
    writer.add_scalar("Accuracy/val",   val_acc,   epoch)
    writer.add_scalar("LR", scheduler.get_last_lr()[0], epoch)

    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
          f"Val Loss:   {val_loss:.4f},   Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save_pretrained(SAVE_PATH)
        feature_extractor.save_pretrained(SAVE_PATH)
        print(f"  ✓ New best saved (val_acc={val_acc:.4f})")

print(f"\nBest val acc: {best_val_acc:.4f}")

### **Evaluation**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
def evaluate(model, loader, device, threshold=0.5):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    with torch.no_grad():
        for batch in loader:
            x, y = batch[0], batch[1]
            x = x.to(device)
            y = y.to(device)

            output = model(x)                              # ← no .unsqueeze(0)
            logits = output.logits                         # ← extract tensor from ImageClassifierOutput
            probs  = torch.sigmoid(logits).cpu()           # [B, 1]
            preds  = (probs >= threshold).float()

            all_probs.extend(probs.view(-1).tolist())    # flatten to 1D
            all_preds.extend(preds.view(-1).tolist())    # flatten to 1D
            all_labels.extend(y.cpu().view(-1).tolist()) # flatten to 1D            # ← no squeeze needed on labels

    results_df = test_split.copy().reset_index(drop=True)
    results_df["true_label"]   = all_labels
    results_df["pred_prob_ai"] = all_probs
    results_df["pred_label"]   = all_preds
    results_df["correct"] = (results_df["true_label"] == results_df["pred_label"])

    cm = confusion_matrix(results_df["true_label"], results_df["pred_label"])
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["Human (0)", "AI (1)"],
        yticklabels=["Human (0)", "AI (1)"]
    )
    plt.title("AI vs Human Generated Images Confusion Matrix - Validation Set")
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()
    plt.show()
    
    acc = results_df["correct"].mean()
    print(f"Test accuracy: {acc:.4f}")
    print(classification_report(
        results_df["true_label"],
        results_df["pred_label"],
        target_names=["Human (0)", "AI (1)"]
    ))

    if "source_pool" in results_df.columns:
        print("\nAccuracy by source_pool:")
        print(results_df.groupby("source_pool")["correct"].mean().sort_values(ascending=False))

    return results_df

results_df = evaluate(model, test_loader, device)

In [ ]:
def evau_image(loader, df):
    model.to(device)
    model.eval()
    
    all_probs = []
    all_preds = []
    all_true = []
    
    with torch.no_grad():
        for pixel_values, labels in loader:
            pixel_values = pixel_values.to(device)
            labels = labels.float().squeeze(1).to(device)
    
            outputs = model(pixel_values=pixel_values)
            logits = outputs.logits.squeeze(1)
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.3).long()  # using your threshold of 0.3
    
            all_probs.extend(probs.cpu().tolist())
            all_preds.extend(preds.cpu().tolist())
            all_true.extend(labels.cpu().tolist())
    
    val_results_df = df.copy().reset_index(drop=True)
    val_results_df["true_label"] = [int(x) for x in all_true]
    val_results_df["pred_prob_ai"] = all_probs
    val_results_df["pred_label"] = all_preds
    val_results_df["correct"] = (
        val_results_df["true_label"] == val_results_df["pred_label"]
    )
    
    display(val_results_df.head(20))
    return val_results_df

In [ ]:
cm = confusion_matrix(results_df["true_label"], results_df["pred_label"])
disp = ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=["Human (0)", "AI (1)"]
        ) 
fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
plt.title("AI vs Human Generated Images Confusion Matrix - Validation Set")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.show()

# Overall metrics
acc = results_df["correct"].mean()
print(f"Test accuracy: {acc:.4f}")
print(classification_report(
    results_df["true_label"],
    results_df["pred_label"],
    target_names=["Human (0)", "AI (1)"]
))

In [ ]:
gan_df, gan_images = images(folder[0], "1", "gan")
mj_df, mj_images = images(folder[1], "1", "mj")
real_df, real_images = images(folder[2], "0", "real")
sd_df, sd_images = images(folder[3], "1", "sd")
pool_df = pd.concat([gan_df,mj_df, real_df, sd_df])
pool_df.head()

In [ ]:
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
BATCH_SIZE = 32
NUM_WORKERS = 0
dalle_folder = ["/kaggle/input/datasets/aryankaushik005/dalleg/dalle/airplane/*.jpg",
                "/kaggle/input/datasets/aryankaushik005/dalleg/dalle/car/*.jpg",
                "/kaggle/input/datasets/aryankaushik005/dalleg/dalle/cat/*.jpg",
                "/kaggle/input/datasets/aryankaushik005/dalleg/dalle/dog/*.jpg"
]
val_tfms = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
dalle_dfs = pd.DataFrame(columns=["file_name", "label"])
for i in dalle_folder:
    df = images(i, "1", "dalle")
    dalle_dfs = pd.concat([dalle_dfs, df[0]])
dalles_ds = BinaryImageDataset(dalle_dfs,   transform=val_tfms,   is_test=False)
dalles_val_loader = DataLoader(
    dalles_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
x, y = next(iter(dalles_val_loader))
print("x:", x.shape, x.dtype)   # [B, 3, 224, 224]
print("y:", y.shape, y.dtype)   # [B, 1], float32
print("y unique (sample):", torch.unique(y)[:10])

In [ ]:
dalle_result = evau_image(dalles_val_loader, dalle_dfs)
gan_result = evau_image(gan_images, gan_df)
mj_result = evau_image(mj_images, mj_df)
real_result = evau_image(real_images, real_df)
sd_result = evau_image(sd_images, sd_df)
pool_result = evau_image(pool_images, pool_df)

In [ ]:
def matrix(df):
    cm = confusion_matrix(
        df["true_label"],
        df["pred_label"]
    )

    # Display raw matrix values
    print(cm)
    
    
    # Plot confusion matrix
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Human (0)", "AI (1)"]
    )
    
    fig, ax = plt.subplots(figsize=(6, 6))
    disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    plt.title("AI vs Human Generated Images Confusion Matrix - Dalle Set")
    plt.show()
    return cm

def con(cm):
    print(cm)
    disp = ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=["Human (0)", "AI (1)"]
        )
        
    fig, ax = plt.subplots(figsize=(6, 6))
    disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    plt.title("AI vs Human Generated Training Set Testing Confusion Matrix - Testing Set")
    plt.show()

In [ ]:
con(gen)
con(value)
value = matrix(dalles_result)
value = matrix(dalle_result)
gen = matrix(gan_result)
gen += matrix(mj_result)
gen += matrix(real_result)
gen += matrix(sd_result)
print(gen)

### **Save VIA Huggingface**

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Login
secrets = UserSecretsClient()
hf_token = secrets.get_secret("Model_Writer")
login(token=hf_token)

# Push everything to HF
model.push_to_hub("gazeng/vit-ai-detector_3")
feature_extractor.push_to_hub("gazeng/vit-ai-detector_3")

print("Done! Check https://huggingface.co/your-username/vit-ai-detector")